# EEG-Conformer + UDA-DANN 跨试次分类

本 notebook 在`eegconformer.ipynb`的卷积Patch Embedding与Transformer Encoder基础上增加DANN域适应：

```text
source train: test_2～test_7（EEG + 频率标签）
source test:  test_8（独立测试）
target UDA:   four_1～four_4（训练时只有EEG）
drift test:   four_5（最终跨试次测试）

EEG → EEG-Conformer feature extractor
                       ├─ class head：15/12/10 Hz（source监督）
                       └─ GRL → domain head：source/target（域对抗）
```

实验协议：

- `test_2～test_7`组成source训练池，其中20%只用于source validation；`test_8`是独立source测试。
- `four_1～four_4`的EEG无标签参与DANN训练；`four_5`完全不参与训练。
- `test_*`使用完整试次；`four_*`按250 Hz取0.5～1.5 s，即样本`[125:375]`。
- 最佳epoch只由source validation三分类准确率决定，不使用`test_8`或`four_5`频率标签。
- 对`four_1～four_4`而言这是UDA；`four_5`是训练期间完全不可见的漂移测试。

三分类随机水平约33.3%；域判别器是source/target二分类，其平衡随机水平是50%。


In [5]:
from copy import deepcopy
from itertools import cycle
from pathlib import Path
import math
import random

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


device: cpu


## 1. Source、无标签适配数据与独立测试数据

归一化均值和标准差只由source训练子集计算。适配DataLoader只含`four_1～four_4` EEG，不含频率标签；两个测试集的标签到训练全部结束后才加载。


In [ ]:
DATA_DIR = Path("data")
TRAIN_FILE_PATTERNS = [f"test_{index}*ssvep" for index in range(2, 8)]
TEST_FILE_PATTERNS = ["test_8_ssvep"]
ADAPTATION_FILE_PATTERNS = [f"test_{index}_ssvep" for index in range(11, 15)]
DRIFT_FILE_PATTERNS = ["test_15_ssvep"]
FREQUENCIES_HZ = [15, 12, 10]
BATCH_SIZE = 16
MAX_EPOCHS = 200


def resolve_trial_stems(patterns):
    resolved = []
    for pattern in patterns:
        matches = sorted(DATA_DIR.glob(f"eeg_trials_{pattern}.npy"))
        matches = [path for path in matches if not path.stem.endswith("_mask")]
        if len(matches) != 1:
            raise ValueError(f"{pattern!r} resolved to {len(matches)} files: {matches}")
        resolved.append(matches[0].stem.removeprefix("eeg_trials_"))
    return resolved


TRAIN_FILES = resolve_trial_stems(TRAIN_FILE_PATTERNS)
TEST_FILES = resolve_trial_stems(TEST_FILE_PATTERNS)
ADAPTATION_FILES = resolve_trial_stems(ADAPTATION_FILE_PATTERNS)
DRIFT_FILES = resolve_trial_stems(DRIFT_FILE_PATTERNS)
print("TRAIN_FILES:", TRAIN_FILES)
print("TEST_FILES:", TEST_FILES)
print("ADAPTATION_FILES:", ADAPTATION_FILES)
print("DRIFT_FILES:", DRIFT_FILES)


def load_eeg_file(name, four_window=False):
    x = np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True)
    x = np.stack(x) if np.asarray(x).dtype == object else np.asarray(x)
    if x.ndim != 3:
        raise ValueError(f"{name}: expected EEG=[trial,channel,time]")
    if four_window:
        # if x.shape[-1] < 375:
        #     raise ValueError(f"{name}: fewer than 375 samples; cannot take [0:375]")
        x = x[:, :, 0:375]
    return x.astype(np.float32)


def load_labeled_files(file_names):
    trials, labels = [], []
    for name in file_names:
        x = load_eeg_file(name)
        y = np.load(DATA_DIR / f"eeg_labels_{name}.npy").astype(np.int64)
        if len(x) != len(y):
            raise ValueError(f"{name}: expected X=[trial,channel,time] with matching labels")
        if not np.isin(y, np.arange(len(FREQUENCIES_HZ))).all():
            raise ValueError(f"{name}: labels must be 0..{len(FREQUENCIES_HZ)-1}")
        trials.append(x)
        labels.append(y)
        print(name, x.shape, y.shape, np.bincount(y, minlength=len(FREQUENCIES_HZ)))
    return trials, labels


source_arrays, source_label_arrays = load_labeled_files(TRAIN_FILES)
source_samples = min(x.shape[-1] for x in source_arrays)
source_arrays = [x[:, :, :source_samples] for x in source_arrays]

# Only EEG is loaded for adaptation and both tests at this stage.
X_source_test = np.concatenate([load_eeg_file(name)[:, :, :source_samples] for name in TEST_FILES])
X_target_unlabeled = np.concatenate([
    load_eeg_file(name, four_window=True) for name in ADAPTATION_FILES
])
X_drift_test = np.concatenate([
    load_eeg_file(name, four_window=True) for name in DRIFT_FILES
])
print("source test EEG:", X_source_test.shape)
print("unlabeled adaptation EEG:", X_target_unlabeled.shape)
print("unseen drift test EEG:", X_drift_test.shape)

X_source_pool = np.concatenate(source_arrays).astype(np.float32)
y_source_pool = np.concatenate(source_label_arrays).astype(np.int64)

X_source_train, X_source_validation, y_source_train, y_source_validation = train_test_split(
    X_source_pool, y_source_pool, test_size=0.20,
    random_state=SEED, stratify=y_source_pool,
)

# Source-only normalization: target statistics do not affect the transform.
mean = X_source_train.mean(axis=(0, 2), keepdims=True)
std = X_source_train.std(axis=(0, 2), keepdims=True) + 1e-6
X_source_train = (X_source_train - mean) / std
X_source_validation = (X_source_validation - mean) / std
X_source_test = (X_source_test - mean) / std
X_target_unlabeled = (X_target_unlabeled - mean) / std
X_drift_test = (X_drift_test - mean) / std

X_source_train = torch.tensor(X_source_train, dtype=torch.float32).unsqueeze(1)
X_source_validation = torch.tensor(X_source_validation, dtype=torch.float32).unsqueeze(1)
X_source_test = torch.tensor(X_source_test, dtype=torch.float32).unsqueeze(1)
X_target_unlabeled = torch.tensor(X_target_unlabeled, dtype=torch.float32).unsqueeze(1)
X_drift_test = torch.tensor(X_drift_test, dtype=torch.float32).unsqueeze(1)
y_source_train = torch.tensor(y_source_train, dtype=torch.long)
y_source_validation = torch.tensor(y_source_validation, dtype=torch.long)


def make_source_train_loader(seed):
    return DataLoader(
        TensorDataset(X_source_train, y_source_train),
        batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )


def make_target_train_loader(seed):
    # This dataset contains no SSVEP class labels.
    return DataLoader(
        TensorDataset(X_target_unlabeled),
        batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )


source_validation_loader = DataLoader(
    TensorDataset(X_source_validation, y_source_validation),
    batch_size=64, shuffle=False,
)
print("source samples:", source_samples, "four samples:", X_target_unlabeled.shape[-1])
print("source train / validation:", len(y_source_train), len(y_source_validation))
print("source test / adaptation / drift test:", len(X_source_test), len(X_target_unlabeled), len(X_drift_test))


TRAIN_FILES: ['test_2_ssvep', 'test_3_ssvep', 'test_4_ssvep', 'test_5_ssvep', 'test_6_ssvep', 'test_7_ssvep']
TEST_FILES: ['test_8_ssvep']
ADAPTATION_FILES: ['test_11_ssvep', 'test_12_ssvep', 'test_13_ssvep', 'test_14_ssvep']
DRIFT_FILES: ['test_15_ssvep']
test_2_ssvep (19, 8, 374) (19,) [8 4 7]
test_3_ssvep (41, 8, 373) (41,) [11 16 14]
test_4_ssvep (69, 8, 374) (69,) [17 23 29]
test_5_ssvep (102, 8, 373) (102,) [30 35 37]
test_6_ssvep (106, 8, 373) (106,) [17 43 46]
test_7_ssvep (67, 8, 373) (67,) [19 20 28]


ValueError: test_11_ssvep: fewer than 375 samples; cannot take [125:375]

## 2. EEG-Conformer、GRL与两个分类头

`class_head`预测三种频率；`domain_head`只判断source或target。GRL前向不改变特征，反向把域损失梯度乘以负的`alpha`，迫使特征提取器削弱试次域信息。


In [ ]:
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, features, alpha):
        ctx.alpha = alpha
        return features.view_as(features)

    @staticmethod
    def backward(ctx, gradient):
        return -ctx.alpha * gradient, None


class GradientReversal(nn.Module):
    def forward(self, features, alpha=1.0):
        return GradientReversalFunction.apply(features, alpha)


class PatchEmbedding(nn.Module):
    def __init__(self, channels, embedding_size=40, dropout=0.5):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Conv2d(1, embedding_size, kernel_size=(1, 25), bias=False),
            nn.Conv2d(
                embedding_size, embedding_size,
                kernel_size=(channels, 1), bias=False,
            ),
            nn.BatchNorm2d(embedding_size),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 75), stride=(1, 15)),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.projection(x).squeeze(2).transpose(1, 2)


class EEGConformerDANN(nn.Module):
    def __init__(
        self, channels, samples, classes=3, embedding_size=40,
        transformer_depth=6, attention_heads=10,
        feedforward_expansion=4, dropout=0.5,
    ):
        super().__init__()
        self.patch_embedding = PatchEmbedding(channels, embedding_size, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_size,
            nhead=attention_heads,
            dim_feedforward=embedding_size * feedforward_expansion,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=transformer_depth,
            enable_nested_tensor=False,
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 1, channels, samples)
            feature_size = self.extract_features(dummy).shape[1]
        self.class_head = nn.Sequential(
            nn.Linear(feature_size, 256), nn.ELU(), nn.Dropout(0.5),
            nn.Linear(256, 32), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(32, classes),
        )
        self.grl = GradientReversal()
        self.domain_head = nn.Sequential(
            nn.Linear(feature_size, 128), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(128, 2),
        )

    def extract_features(self, x):
        tokens = self.patch_embedding(x)
        tokens = self.transformer(tokens)
        # Mean pooling keeps the feature size fixed for full test trials and
        # the 0.5--1.5 s four_* windows without padding or time resampling.
        return tokens.mean(dim=1)

    def classify(self, x):
        return self.class_head(self.extract_features(x))

    def forward(self, x, alpha=1.0, with_domain=True):
        features = self.extract_features(x)
        class_logits = self.class_head(features)
        if not with_domain:
            return class_logits
        domain_logits = self.domain_head(self.grl(features, alpha))
        return class_logits, domain_logits


torch.manual_seed(SEED)
template_model = EEGConformerDANN(
    channels=X_source_train.shape[2], samples=X_source_train.shape[-1],
    classes=len(FREQUENCIES_HZ),
)
initial_state = deepcopy(template_model.state_dict())
with torch.no_grad():
    dummy = torch.zeros(2, 1, X_source_train.shape[2], X_source_train.shape[-1])
    dummy_class, dummy_domain = template_model(dummy)
    short_dummy = torch.zeros(2, 1, X_source_train.shape[2], X_target_unlabeled.shape[-1])
    short_class, short_domain = template_model(short_dummy)
print("class output:", tuple(dummy_class.shape), "domain output:", tuple(dummy_domain.shape))
print("four-window output:", tuple(short_class.shape), tuple(short_domain.shape))
print("trainable parameters:", sum(p.numel() for p in template_model.parameters()))
assert dummy_class.shape == (2, 3)
assert dummy_domain.shape == (2, 2)
assert short_class.shape == (2, 3)
assert short_domain.shape == (2, 2)


class output: (2, 3) domain output: (2, 2)
four-window output: (2, 3) (2, 2)
trainable parameters: 156525


## 3. 训练UDA-DANN

只训练一次DANN，并且只根据source validation三分类准确率保存最佳epoch。DANN训练中没有任何`four_*`频率标签。


In [ ]:
CLASS_LOSS = nn.CrossEntropyLoss()
DOMAIN_LOSS = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate_source_validation(model):
    model.eval()
    predictions, labels = [], []
    for eeg, target in source_validation_loader:
        logits = model.classify(eeg.to(DEVICE))
        predictions.append(logits.argmax(1).cpu())
        labels.append(target)
    prediction = torch.cat(predictions).numpy()
    label = torch.cat(labels).numpy()
    return float(np.mean(prediction == label))


def dann_training():
    torch.manual_seed(SEED)
    model = EEGConformerDANN(
        X_source_train.shape[2], X_source_train.shape[-1], len(FREQUENCIES_HZ)
    ).to(DEVICE)
    model.load_state_dict(initial_state)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    source_loader = make_source_train_loader(SEED)
    target_loader = make_target_train_loader(SEED)
    target_iterator = cycle(target_loader)
    total_steps = MAX_EPOCHS * len(source_loader)
    global_step = 0
    best_accuracy, best_epoch, best_state = -1.0, None, None
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        class_sum = domain_sum = 0.0
        for source_eeg, source_class in source_loader:
            target_eeg_batch = next(target_iterator)[0]
            source_eeg = source_eeg.to(DEVICE)
            source_class = source_class.to(DEVICE)
            target_eeg_batch = target_eeg_batch.to(DEVICE)

            progress = global_step / max(total_steps - 1, 1)
            alpha = 2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0
            source_class_logits, source_domain_logits = model(source_eeg, alpha)
            _, target_domain_logits = model(target_eeg_batch, alpha)
            source_domain = torch.zeros(len(source_eeg), dtype=torch.long, device=DEVICE)
            target_domain = torch.ones(len(target_eeg_batch), dtype=torch.long, device=DEVICE)

            class_loss = CLASS_LOSS(source_class_logits, source_class)
            domain_loss = DOMAIN_LOSS(
                torch.cat([source_domain_logits, target_domain_logits]),
                torch.cat([source_domain, target_domain]),
            )
            loss = class_loss + domain_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            class_sum += class_loss.item() * len(source_class)
            domain_sum += domain_loss.item()
            global_step += 1

        validation_accuracy = evaluate_source_validation(model)
        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_epoch = epoch
            best_state = deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        if epoch == 1 or epoch % 20 == 0:
            print(
                f"DANN epoch={epoch:03d} class_loss={class_sum / len(source_loader.dataset):.4f} "
                f"domain_loss={domain_sum / len(source_loader):.4f} alpha={alpha:.4f} "
                f"source_val={validation_accuracy:.4f} best={best_accuracy:.4f}"
            )
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, best_epoch, best_accuracy


dann_model, dann_best_epoch, dann_source_validation = dann_training()
print("DANN selected:", dann_best_epoch, dann_source_validation)


DANN epoch=001 class_loss=1.1397 domain_loss=0.7010 alpha=0.0238 source_val=0.3704 best=0.3704
DANN epoch=020 class_loss=0.8036 domain_loss=0.6919 alpha=0.4613 source_val=0.4691 best=0.5309
DANN epoch=040 class_loss=0.6581 domain_loss=0.6960 alpha=0.7612 source_val=0.3951 best=0.5309
DANN epoch=060 class_loss=0.4535 domain_loss=0.6901 alpha=0.9050 source_val=0.3951 best=0.5309
DANN epoch=080 class_loss=0.2960 domain_loss=0.6768 alpha=0.9640 source_val=0.4074 best=0.5309
DANN epoch=100 class_loss=0.1684 domain_loss=0.6775 alpha=0.9866 source_val=0.4074 best=0.5309
DANN epoch=120 class_loss=0.3073 domain_loss=0.6837 alpha=0.9951 source_val=0.3951 best=0.5926
DANN epoch=140 class_loss=0.1680 domain_loss=0.6835 alpha=0.9982 source_val=0.3951 best=0.5926
DANN epoch=160 class_loss=0.2919 domain_loss=0.6827 alpha=0.9993 source_val=0.3951 best=0.6420
DANN epoch=180 class_loss=0.4518 domain_loss=0.6924 alpha=0.9998 source_val=0.3951 best=0.6420
DANN epoch=200 class_loss=0.2516 domain_loss=0.709

## 4. 域判别结果、Source测试与跨试次漂移测试

到这里才读取`test_8`和`four_5`的频率标签。域判别器的平衡准确率只表示source与`four_1～four_4`是否仍可区分，不是三分类准确率。最终SSVEP预测只使用`class_head`。


In [ ]:
@torch.no_grad()
def domain_balanced_accuracy(model):
    model.eval()
    source_correct = source_total = target_correct = target_total = 0
    for eeg, _ in source_validation_loader:
        features = model.extract_features(eeg.to(DEVICE))
        prediction = model.domain_head(features).argmax(1).cpu()
        source_correct += int((prediction == 0).sum())
        source_total += len(prediction)
    target_loader = DataLoader(TensorDataset(X_target_unlabeled), batch_size=64)
    for (eeg,) in target_loader:
        features = model.extract_features(eeg.to(DEVICE))
        prediction = model.domain_head(features).argmax(1).cpu()
        target_correct += int((prediction == 1).sum())
        target_total += len(prediction)
    source_recall = source_correct / source_total
    target_recall = target_correct / target_total
    return (source_recall + target_recall) / 2, source_recall, target_recall


@torch.no_grad()
def target_predictions(model, loader):
    model.eval()
    predictions, labels = [], []
    for eeg, target in loader:
        predictions.append(model.classify(eeg.to(DEVICE)).argmax(1).cpu())
        labels.append(target)
    return torch.cat(labels).numpy(), torch.cat(predictions).numpy()


# Test labels are loaded only after DANN training finishes.
def load_final_labels(file_names):
    return np.concatenate([
        np.load(DATA_DIR / f"eeg_labels_{name}.npy").astype(np.int64)
        for name in file_names
    ])


y_source_test_final = load_final_labels(TEST_FILES)
y_drift_test_final = load_final_labels(DRIFT_FILES)
if len(y_source_test_final) != len(X_source_test):
    raise ValueError("source test EEG and labels have different lengths")
if len(y_drift_test_final) != len(X_drift_test):
    raise ValueError("drift test EEG and labels have different lengths")
source_test_loader = DataLoader(
    TensorDataset(X_source_test, torch.tensor(y_source_test_final, dtype=torch.long)),
    batch_size=64, shuffle=False,
)
drift_test_loader = DataLoader(
    TensorDataset(X_drift_test, torch.tensor(y_drift_test_final, dtype=torch.long)),
    batch_size=64, shuffle=False,
)

domain_balanced, source_domain_recall, target_domain_recall = domain_balanced_accuracy(dann_model)
print(
    f"DANN domain-head balanced accuracy: {domain_balanced:.4f} "
    f"(source recall={source_domain_recall:.4f}, target recall={target_domain_recall:.4f})"
)

for test_name, loader in (("test_8", source_test_loader), ("four_5 drift", drift_test_loader)):
    y_true, y_pred = target_predictions(dann_model, loader)
    accuracy = np.mean(y_true == y_pred)
    print(f"\nUDA-DANN Conformer {test_name} accuracy: {accuracy:.4f} ({np.sum(y_true == y_pred)}/{len(y_true)})")
    for label, frequency in enumerate(FREQUENCIES_HZ):
        mask = y_true == label
        recall = np.mean(y_pred[mask] == label)
        print(f"{frequency} Hz recall: {recall:.4f} ({np.sum(y_pred[mask] == label)}/{np.sum(mask)})")
    print("confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]")
    print(confusion_matrix(y_true, y_pred, labels=np.arange(len(FREQUENCIES_HZ))))
    print(classification_report(
        y_true, y_pred,
        labels=np.arange(len(FREQUENCIES_HZ)),
        target_names=[f"{frequency} Hz" for frequency in FREQUENCIES_HZ],
        digits=4, zero_division=0,
    ))


DANN domain-head balanced accuracy: 0.5843 (source recall=1.0000, target recall=0.1687)

UDA-DANN Conformer test_8 accuracy: 0.4925 (33/67)
15 Hz recall: 0.0000 (0/23)
12 Hz recall: 0.6667 (16/24)
10 Hz recall: 0.8500 (17/20)
confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]
[[ 0  8 15]
 [ 0 16  8]
 [ 0  3 17]]
              precision    recall  f1-score   support

       15 Hz     0.0000    0.0000    0.0000        23
       12 Hz     0.5926    0.6667    0.6275        24
       10 Hz     0.4250    0.8500    0.5667        20

    accuracy                         0.4925        67
   macro avg     0.3392    0.5056    0.3980        67
weighted avg     0.3391    0.4925    0.3939        67


UDA-DANN Conformer four_5 drift accuracy: 0.2960 (37/125)
15 Hz recall: 0.8000 (20/25)
12 Hz recall: 0.3409 (15/44)
10 Hz recall: 0.0357 (2/56)
confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]
[[20  4  1]
 [29 15  0]
 [48  6  2]]
              precision    recall  f